In [1]:
from typing import Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

c:\Users\yazan\anaconda3\envs\ml-rag-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. first I will define the shared data it will tell langgraph what information every agent can acess.
# Every node or agent will receive this dictionary
class AgentState(TypedDict):
    # Without TypedDict, Python only knows something is a dictionary without any constraints
    # With TypedDict, This tells Python: "Any dictionary of type AgentState must have these keys and these value types."
    question: str
    answer: str
    vaild: bool

In [26]:
# 2. Create agent 1

def answer_agent(state: AgentState): # the agent will receive a state (which contains the entire shared data) of type AgentState

    # get the question from the state 
    question = state["question"]

    # Generate an answer
    answer = f"The answer to '{question}' is generated by the AI."


    # Fill the dict
    return {
        "answer": answer
    }


In [27]:
# Create Agent 2 

def validation_agent(state: AgentState):
    
    # Get the answer
    answer = state["answer"]


    # Fill the dic

    if answer:

        return {
            "valid": True
        }

    else:

        return {
            "valid": False
        }



In [28]:
# Create the graph

graph = StateGraph(AgentState)

In [29]:
# Add the node / agents

# add_node() takes 2 arguments: 1- name of the node / agnet example here: "answer_agent"
# 2- is the agent method, no () because we do not want to run it now, langgraph only want to remember which function to call later when its turn.
graph.add_node(
    "answer_agent", answer_agent
)


graph.add_node(
    "validation_agent", validation_agent
)


# Connect the nodes from left to right, so here you tell LangGraph which agent to go after this agent, so we tell the order

graph.add_edge(
    "answer_agent", "validation_agent"
)


### Now we need to define where the graph starts and where it stops.

Your graph currently has two nodes:
```
answer_agent
      |
      |
validation_agent
```
But LangGraph needs to know:

Where do I begin?
When am I finished?

In [30]:
# 1. Define the start point

graph.set_entry_point(
    "answer_agent"
    )

# This tells LangGraph:
# "When someone runs this graph, start by executing the node called answer_agent."

# The graph becomes:

# START
#   |
#   ▼
# answer_agent
#   |
#   ▼
# validation_agent

# 2. Define the ending

graph.add_edge(
    "validation_agent",
    END
)
# This says:
# "After the validation_agent finishes, stop the graph."



In [31]:
# the code up we only defined and descibed the workflow. 
# Now compile the graph


app = graph.compile() # compile() converts your blueprint into a runnable object.
# Now app can receive input and execute the agents.


# Run graph
result = app.invoke(
    {
        "question": "What is YOLO?",
        "answer": "",
        "valid": False
    }
)

print(result)


{'question': 'What is YOLO?', 'answer': "The answer to 'What is YOLO?' is generated by the AI."}
